# مسئلهٔ ۱ — V2 / تحلیل خطای A2 در full-MP4 inference

این نوت‌بوک خروجی video-level مدل A2 را تحلیل می‌کند. هیچ مدلی آموزش داده نمی‌شود و metadata فقط برای شناخت الگوی خطا به کار می‌رود، نه به‌عنوان feature مدل.

چهار گروه True Positive، True Negative، False Positive و False Negative ساخته می‌شوند. برای خطاهای مهم montage فریم‌های پنجرهٔ با بیشترین احتمال ذخیره می‌شود.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math

import cv2
import numpy as np
import pandas as pd

DATA_ROOT = Path(r'P:\NexarCollisionData')
VIDEO_MANIFEST_PATH = DATA_ROOT / 'video_manifest_v2.csv'
INFERENCE_DIR = DATA_ROOT / 'inference_v2'
VIDEO_PREDICTIONS_PATH = INFERENCE_DIR / 'a2_validation_sliding_video_predictions.csv'
WINDOW_PREDICTIONS_PATH = INFERENCE_DIR / 'a2_validation_sliding_window_predictions.csv'
METRICS_PATH = INFERENCE_DIR / 'a2_validation_sliding_metrics.json'
ERROR_ANALYSIS_PATH = INFERENCE_DIR / 'a2_full_video_error_analysis.csv'
METADATA_BREAKDOWN_PATH = INFERENCE_DIR / 'a2_full_video_error_by_metadata.csv'
SUMMARY_PATH = INFERENCE_DIR / 'a2_full_video_error_summary.json'
ERROR_IMAGE_DIR = INFERENCE_DIR / 'error_analysis'
ERROR_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

MAX_EXAMPLES_PER_ERROR_TYPE = 12
FRAMES_PER_VISUALIZATION = 4

assert VIDEO_MANIFEST_PATH.exists(), 'Run notebook 07 first.'
assert VIDEO_PREDICTIONS_PATH.exists(), 'Run notebook 16 first.'
assert WINDOW_PREDICTIONS_PATH.exists(), 'Run notebook 16 first.'
assert METRICS_PATH.exists(), 'Run notebook 16 first.'

In [2]:
metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
selected_threshold = float(metrics['selected_threshold_by_validation_f1'])
selected_aggregation = str(metrics['selected_aggregation'])

manifest = pd.read_csv(VIDEO_MANIFEST_PATH).copy()
video_predictions = pd.read_csv(VIDEO_PREDICTIONS_PATH).copy()
window_predictions = pd.read_csv(WINDOW_PREDICTIONS_PATH).copy()
for table in (manifest, video_predictions, window_predictions):
    table['video_id'] = table['video_id'].astype(str)
manifest['time_of_event'] = pd.to_numeric(manifest['time_of_event'], errors='coerce')
video_predictions['label'] = video_predictions['label'].astype(int)
video_predictions['prediction'] = video_predictions['prediction'].astype(int)

metadata_columns = ['video_id', 'weather', 'light_conditions', 'scene', 'time_of_event']
analysis = video_predictions.merge(manifest[metadata_columns], on='video_id', how='left', validate='one_to_one', suffixes=('', '_manifest'))
assert len(analysis) == 120
assert analysis['label'].isin([0, 1]).all()
assert (analysis['prediction'] == (analysis['video_probability'] >= selected_threshold).astype(int)).all()

def error_type(row: pd.Series) -> str:
    if row.label == 1 and row.prediction == 1:
        return 'true_positive'
    if row.label == 0 and row.prediction == 0:
        return 'true_negative'
    if row.label == 0 and row.prediction == 1:
        return 'false_positive'
    return 'false_negative'

analysis['error_type'] = analysis.apply(error_type, axis=1)
analysis['is_error'] = analysis['error_type'].isin(['false_positive', 'false_negative'])
analysis['max_window_contains_event'] = (
    analysis['time_of_event'].notna()
    & analysis['time_of_event'].between(analysis['max_window_start'], analysis['max_window_end'])
)
analysis['max_window_center_abs_error_seconds'] = np.where(
    analysis['time_of_event'].notna(),
    (analysis['time_of_event'] - analysis['max_window_center']).abs(),
    np.nan,
)
analysis = analysis.sort_values(['is_error', 'video_probability'], ascending=[False, False]).reset_index(drop=True)
analysis.to_csv(ERROR_ANALYSIS_PATH, index=False)

print(f'Selected aggregation: {selected_aggregation}; threshold: {selected_threshold:.2f}')
display(analysis['error_type'].value_counts().rename_axis('error_type').to_frame('videos'))
display(analysis.loc[analysis['is_error'], ['video_id', 'error_type', 'label', 'prediction', 'video_probability', 'max_window_start', 'max_window_end', 'time_of_event']].head(20))

Selected aggregation: max; threshold: 0.57


,videos
error_type,
true_positive,52
true_negative,31
false_positive,29
false_negative,8


,video_id,error_type,label,prediction,video_probability,max_window_start,max_window_end,time_of_event
0,1904,false_positive,0,1,0.956838,27.5,32.5,NaN
1,1497,false_positive,0,1,0.947365,17.5,22.5,NaN
2,1870,false_positive,0,1,0.914091,0.0,5.0,NaN
3,1722,false_positive,0,1,0.904963,7.5,12.5,NaN
4,1844,false_positive,0,1,0.889896,32.5,37.5,NaN
5,1959,false_positive,0,1,0.889057,20.0,25.0,NaN
6,1271,false_positive,0,1,0.880514,10.0,15.0,NaN
7,1342,false_positive,0,1,0.870657,20.0,25.0,NaN
8,1673,false_positive,0,1,0.859835,20.0,25.0,NaN
9,1080,false_positive,0,1,0.846718,15.0,20.0,NaN


In [3]:
metadata_rows = []
for column in ('weather', 'light_conditions', 'scene'):
    grouped = analysis.groupby(column, dropna=False)
    for value, group in grouped:
        metadata_rows.append({
            'metadata_field': column,
            'metadata_value': 'missing' if pd.isna(value) else value,
            'videos': int(len(group)),
            'false_positive': int(group['error_type'].eq('false_positive').sum()),
            'false_negative': int(group['error_type'].eq('false_negative').sum()),
            'errors': int(group['is_error'].sum()),
            'error_rate': float(group['is_error'].mean()),
        })
metadata_breakdown = pd.DataFrame(metadata_rows).sort_values(['metadata_field', 'error_rate', 'videos'], ascending=[True, False, False])
metadata_breakdown.to_csv(METADATA_BREAKDOWN_PATH, index=False)

positive_analysis = analysis.loc[analysis['label'].eq(1)].copy()
summary = {
    'model': metrics['model'],
    'selected_aggregation': selected_aggregation,
    'selected_threshold': selected_threshold,
    'true_positive': int(analysis['error_type'].eq('true_positive').sum()),
    'true_negative': int(analysis['error_type'].eq('true_negative').sum()),
    'false_positive': int(analysis['error_type'].eq('false_positive').sum()),
    'false_negative': int(analysis['error_type'].eq('false_negative').sum()),
    'positive_max_window_contains_event_rate': float(positive_analysis['max_window_contains_event'].mean()),
    'positive_max_window_center_mae_seconds': float(positive_analysis['max_window_center_abs_error_seconds'].mean()),
    'note': 'Localization fields are supplementary diagnostics only; the official task is video-level accident detection.',
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Error analysis: {ERROR_ANALYSIS_PATH}')
print(f'Metadata breakdown: {METADATA_BREAKDOWN_PATH}')
print(f'Summary: {SUMMARY_PATH}')
display(metadata_breakdown.head(20))

Error analysis: P:\NexarCollisionData\inference_v2\a2_full_video_error_analysis.csv
Metadata breakdown: P:\NexarCollisionData\inference_v2\a2_full_video_error_by_metadata.csv
Summary: P:\NexarCollisionData\inference_v2\a2_full_video_error_summary.json


,metadata_field,metadata_value,videos,false_positive,false_negative,errors,error_rate
3,light_conditions,Dark,3,0,1,1,0.333333
4,light_conditions,Normal,108,27,7,34,0.314815
5,light_conditions,Twilight,9,2,0,2,0.222222
8,scene,Other,3,1,1,2,0.666667
6,scene,Highway,37,8,4,12,0.324324
10,scene,Urban,53,15,1,16,0.301887
9,scene,Sub-urban,25,5,2,7,0.280000
7,scene,Industrial,2,0,0,0,0.000000
1,weather,Cloudy,44,13,3,16,0.363636
2,weather,Rain,9,3,0,3,0.333333


In [4]:
def read_bgr_at_timestamp(cap: cv2.VideoCapture, timestamp: float, fps: float):
    step = 1.0 / fps if fps > 0 else 1.0 / 30.0
    for offset in (0.0, step, -step):
        cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, timestamp + offset) * 1000.0)
        ok, image = cap.read()
        if ok and image is not None:
            return image
    return None

def video_tile(row: pd.Series, tile_width: int = 640, tile_height: int = 140) -> np.ndarray:
    tile = np.full((tile_height, tile_width, 3), 24, dtype=np.uint8)
    cap = cv2.VideoCapture(str(row.video_path))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) if cap.isOpened() else 0.0
    timestamps = np.linspace(float(row.max_window_start), float(row.max_window_end), num=FRAMES_PER_VISUALIZATION, endpoint=False)
    thumbnail_width, thumbnail_height = 160, 90
    for index, timestamp in enumerate(timestamps):
        image = read_bgr_at_timestamp(cap, float(timestamp), fps) if cap.isOpened() else None
        if image is None:
            image = np.full((thumbnail_height, thumbnail_width, 3), 80, dtype=np.uint8)
        else:
            image = cv2.resize(image, (thumbnail_width, thumbnail_height), interpolation=cv2.INTER_AREA)
        left = index * thumbnail_width
        tile[32:32 + thumbnail_height, left:left + thumbnail_width] = image
    cap.release()
    caption = f"{row.error_type}  id={row.video_id}  p={row.video_probability:.2f}  window={row.max_window_start:.1f}-{row.max_window_end:.1f}s"
    cv2.putText(tile, caption, (8, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (235, 235, 235), 1, cv2.LINE_AA)
    return tile

def make_montage(error_name: str, rows: pd.DataFrame, path: Path, columns: int = 3) -> None:
    selected_rows = rows.head(MAX_EXAMPLES_PER_ERROR_TYPE).reset_index(drop=True)
    if selected_rows.empty:
        return
    tiles = [video_tile(row) for _, row in selected_rows.iterrows()]
    blank = np.full_like(tiles[0], 24)
    while len(tiles) % columns:
        tiles.append(blank.copy())
    montage_rows = [cv2.hconcat(tiles[index:index + columns]) for index in range(0, len(tiles), columns)]
    montage = cv2.vconcat(montage_rows)
    cv2.imwrite(str(path), montage, [cv2.IMWRITE_JPEG_QUALITY, 95])
    print(f'{error_name} montage: {path}')

false_positives = analysis.loc[analysis['error_type'].eq('false_positive')].sort_values('video_probability', ascending=False)
false_negatives = analysis.loc[analysis['error_type'].eq('false_negative')].sort_values('video_probability', ascending=True)
make_montage('False positive', false_positives, ERROR_IMAGE_DIR / 'a2_false_positive_montage.jpg')
make_montage('False negative', false_negatives, ERROR_IMAGE_DIR / 'a2_false_negative_montage.jpg')

False positive montage: P:\NexarCollisionData\inference_v2\error_analysis\a2_false_positive_montage.jpg
False negative montage: P:\NexarCollisionData\inference_v2\error_analysis\a2_false_negative_montage.jpg


## شرط پایان این مرحله

پیش از هر تغییر جدید باید false positiveها و false negativeها را به‌صورت بصری بررسی کنیم. تنها اگر الگوی مشخصی مانند شب، باران، لرزش، overlay یا موقعیت نامناسب پنجره دیده شد، تغییر داده یا مدل را بر اساس همان شواهد انجام می‌دهیم.